Note : Run inside the linux kernel only

# Linux Incident Response: Disk Is Full

## Incident Overview

A full filesystem is a critical Linux production incident. When available disk
space reaches zero, applications may be unable to write logs, temporary files,
database transactions, caches, checkpoints, or uploaded data.

This can result in:

- Application crashes or unexpected failures
- Database write failures or service outages
- Docker containers failing to start
- CI/CD pipelines failing
- AI/ML training jobs stopping because checkpoints cannot be written
- Log rotation failures
- SSH login or system service failures in severe cases

The objective of this incident response procedure is to:

1. Confirm which filesystem is full.
2. Identify what is consuming disk space.
3. Safely reclaim storage.
4. Verify that affected services recover.
5. Prevent the issue from happening again.

> **Important:** Do not immediately delete files. First identify the filesystem,
> understand what is consuming the space, and confirm that the files are safe
> to remove.

---

# 1. Confirm the Problem

## Objective

Verify that the incident is actually caused by disk exhaustion and identify
which filesystem is affected.

### Check Filesystem Usage


In [ ]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
none            7.8G     0  7.8G   0% /usr/lib/modules/6.18.33.2-microsoft-standard-WSL2
none            7.8G  4.0K  7.8G   1% /mnt/wsl
drivers         757G  319G  438G  43% /usr/lib/wsl/drivers
/dev/sdd       1007G  3.3G  953G   1% /
none            7.8G   80K  7.8G   1% /mnt/wslg
none            7.8G     0  7.8G   0% /usr/lib/wsl/lib
rootfs          7.8G  2.8M  7.8G   1% /init
none            7.8G  860K  7.8G   1% /run
none            7.8G     0  7.8G   0% /run/lock
none            7.8G     0  7.8G   0% /run/shm
none            7.8G   80K  7.8G   1% /mnt/wslg/versions.txt
none            7.8G   80K  7.8G   1% /mnt/wslg/doc
C:\             757G  319G  438G  43% /mnt/c
snapfuse        128K  128K     0 100% /snap/bare/5
snapfuse         74M   74M     0 100% /snap/core22/2292
snapfuse         74M   74M     0 100% /snap/core22/2437
snapfuse         92M   92M     0 100% /snap/gtk-common-themes/1535
snapfuse         49M   49M     0 100% /snap

wsl: Failed to translate 'Z:\home\usama\linux_ai_systems_lab'


# 2. Identify the Target Filesystem

## Objective

Determine exactly which filesystem is experiencing disk pressure and identify
its filesystem type and mount point.

A Linux system may contain multiple filesystems, such as:

- Root filesystem (`/`)
- Home directory (`/home`)
- Application data (`/var`)
- Docker storage
- Mounted data volumes
- Network filesystems
- Temporary filesystems

Before deleting or cleaning anything, identify the specific filesystem that is
full.

---

## Display Filesystem Usage and Type


In [ ]:
!df -hT

Filesystem     Type           Size  Used Avail Use% Mounted on
none           overlay        7.8G     0  7.8G   0% /usr/lib/modules/6.18.33.2-microsoft-standard-WSL2
none           tmpfs          7.8G  4.0K  7.8G   1% /mnt/wsl
drivers        9p             757G  316G  441G  42% /usr/lib/wsl/drivers
/dev/sdd       ext4          1007G  3.1G  953G   1% /
none           tmpfs          7.8G   40K  7.8G   1% /mnt/wslg
none           overlay        7.8G     0  7.8G   0% /usr/lib/wsl/lib
rootfs         rootfs         7.8G  2.8M  7.8G   1% /init
none           tmpfs          7.8G  844K  7.8G   1% /run
none           tmpfs          7.8G     0  7.8G   0% /run/lock
none           tmpfs          7.8G     0  7.8G   0% /run/shm
none           overlay        7.8G   80K  7.8G   1% /mnt/wslg/versions.txt
none           overlay        7.8G   80K  7.8G   1% /mnt/wslg/doc
C:\            9p             757G  316G  441G  42% /mnt/c
snapfuse       fuse.snapfuse  128K  128K     0 100% /snap/bare/5
snapfuse    

wsl: Failed to translate 'Z:\home\usama\linux_ai_systems_lab'


# 3. Drill Down into Directory Size

## Objective

After identifying the filesystem that is running out of space, determine which
top-level directories are consuming the most storage.

Instead of scanning every file on the system, start with a
**depth-constrained scan**. This provides a high-level view of disk usage and
helps you quickly identify where to investigate next.

---

## Scan Top-Level Directories



In [ ]:
!du -xh --max-depth=1 / 2>/dev/null |sort -h

16K	/lost+found
16K	/tmp
2.0G	/var
3.2G	/
3.7M	/etc
32K	/snap
363M	/home
4.0K	/boot
4.0K	/media
4.0K	/mnt
4.0K	/opt
4.0K	/root
4.0K	/srv
8.0K	/Docker
882M	/usr


Traverse into the largest suspect directory (e.g., /var)

In [ ]:
!du -xh --max-depth=1 /var 2>/dev/null |sort 

1.1G	/var/lib
1.3M	/var/backups
12K	/var/tmp
160M	/var/cache
16K	/var/spool
2.0G	/var
4.0K	/var/crash
4.0K	/var/local
4.0K	/var/mail
4.0K	/var/opt
76K	/var/snap
814M	/var/log


# 4. Locate Very Large Files

## Objective

After identifying large directories, search for individual files that may be
responsible for consuming a significant amount of disk space.

Large files are commonly caused by:

- Application log files growing without rotation
- Database dumps
- Docker image layers
- Core dump files
- AI/ML model checkpoints
- Downloaded datasets
- Backup archives
- Temporary files
- Large cache files

A single unexpectedly large file can sometimes consume most of the available
disk space.

---

## Search for Files Larger Than 1 GB



In [ ]:
!find / -type f -size +1G 2> /dev/null

^C


# 5. Identify the Top 20 Largest Files

## Objective

Identify the largest individual files across the target system. This is useful
when disk usage is high and directory-level analysis does not immediately reveal
the specific files responsible.

Large files commonly include:

- Application logs
- Database backups or dumps
- Docker image layers
- Core dump files
- AI/ML model checkpoints
- Large datasets
- Archive files
- Temporary files

---

## Find and Sort All Files by Size



In [ ]:
!find / -type f -printf '%s %p\n' 2> /dev/null | sort -nr | head -20

^C


# 6. Check for Open Deleted Files

## Objective

Identify files that have been deleted from the filesystem but are still
consuming disk space because an active process continues to keep the file open.

This is an important investigation step because `df -h` may report that the
filesystem is full while `du` does not show enough files to explain the missing
disk space.

---

## The Problem: Deleted Does Not Always Mean Disk Space Is Released

On Linux, deleting a file removes its directory entry, but the underlying disk
space is not immediately released if a running process still has the file open.


In [ ]:
!lsof +L1

## Resolution: Release the File Lock

If a file cannot be removed or disk space is not being reclaimed because a
process still has the file open, identify the responsible process and
terminate or restart it gracefully.

### Option 1: Terminate the Process

Use the process ID (`PID`) identified with `lsof`:


In [ ]:
!kill 9998
# or restart the managing service
!sudo systemctl restart <service-name>

# 7. Diagnose Docker Bloat

## Objective

Docker can consume significant disk space over time through:

- Unused image layers
- Dangling images
- Stopped containers
- Build cache
- Unused volumes
- Old application images
- Multiple versions of the same image

Docker commonly stores this data under:

```text
/var/lib/docker

In [ ]:
!docker system df


The command 'docker' could not be found in this WSL 2 distro.
We recommend to activate the WSL integration in Docker Desktop settings.

For details about using Docker Desktop with WSL 2, visit:

https://docs.docker.com/go/wsl2/



wsl: Failed to translate 'Z:\home\usama\linux_ai_systems_lab'


## Cleanup Options

In [ ]:
# Safe prune (unused containers, dangling images, network bridges)
!docker system prune -f

# Aggressive prune (all unused images, stopped containers, build caches)
!docker system prune -a -f

# 8. AI/ML Workload Cleanup

## Objective

AI/ML environments can consume disk space much faster than conventional
applications.

Model weights, Hugging Face caches, training checkpoints, datasets, experiment
artifacts, Docker layers, and Python package caches can accumulate over time.

A single model download can occupy several GB, while repeated downloads of
different model versions can consume tens or hundreds of GB.

Before deleting anything, identify which cache or artifact directory is
responsible for the storage growth.

---

## Common AI/ML Storage Consumers

Typical sources of disk usage include:

| Storage Location | Typical Contents |
|---|---|
| `~/.cache/huggingface` | Hugging Face models, datasets, tokenizers |
| `~/.cache/torch` | PyTorch-related downloaded artifacts |
| `~/.cache/pip` | Downloaded Python packages and wheels |
| `~/.cache` | Application and library caches |
| `~/models` | Downloaded model weights |
| `~/checkpoints` | Training checkpoints |
| `~/datasets` | Local training/evaluation datasets |
| `/tmp` | Temporary inference/training files |
| Docker storage | Images, layers, containers, volumes |
| Project directories | Logs, embeddings, vector indexes, artifacts |

---

# 8.1 Inspect the General User Cache

Start by checking the size of the user's cache directories:



In [ ]:
!du -sh ~/.cache/*
!du -sh ~/.cache/huggingface

Inspect active workspace directories

In [ ]:
!du -sh ~/models ~/checkpoints ~/datasets 2>/dev/null

Clean specific redundant artifacts

In [ ]:
!rm -rf ~/.cache/huggingface/hub/models--*
!rm -f ~/checkpoints/old_checkpoint.pt

# 9. Remediate & Truncate Live Log Files

## Objective

Free disk space from a large log file **without deleting the file itself**.

Deleting a log file that is actively being written to can be misleading. A
running process may still hold the file open through a file descriptor. In that
case, the filename disappears from the filesystem, but the disk space may
remain allocated until the process closes the descriptor.

When immediate space recovery is required and the log file is safe to clear,
**truncate the file to zero bytes instead of deleting it**.

---

## Why Truncation Is Different from Deletion

Consider a running application:

```text
Application
     │
     │ writes
     ▼
application.log

In [ ]:
!sudo truncate -s 0 /var/log/large_application.log

^C


# 10. Verify the Fix & Inode Health
## Confirm space has been reclaimed

In [ ]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
none            7.8G     0  7.8G   0% /usr/lib/modules/6.18.33.2-microsoft-standard-WSL2
none            7.8G  4.0K  7.8G   1% /mnt/wsl
drivers         757G  319G  438G  43% /usr/lib/wsl/drivers
/dev/sdd       1007G  3.3G  953G   1% /
none            7.8G   80K  7.8G   1% /mnt/wslg
none            7.8G     0  7.8G   0% /usr/lib/wsl/lib
rootfs          7.8G  2.8M  7.8G   1% /init
none            7.8G  860K  7.8G   1% /run
none            7.8G     0  7.8G   0% /run/lock
none            7.8G     0  7.8G   0% /run/shm
none            7.8G   80K  7.8G   1% /mnt/wslg/versions.txt
none            7.8G   80K  7.8G   1% /mnt/wslg/doc
C:\             757G  319G  438G  43% /mnt/c
snapfuse        128K  128K     0 100% /snap/bare/5
snapfuse         74M   74M     0 100% /snap/core22/2292
snapfuse         74M   74M     0 100% /snap/core22/2437
snapfuse         92M   92M     0 100% /snap/gtk-common-themes/1535
snapfuse         49M   49M     0 100% /snap

wsl: Failed to translate 'Z:\home\usama\linux_ai_systems_lab'


In [ ]:
!df -i

Filesystem       Inodes   IUsed    IFree IUse% Mounted on
none            2027470       5  2027465    1% /usr/lib/modules/6.18.33.2-microsoft-standard-WSL2
none            2027470       2  2027468    1% /mnt/wsl
drivers             999 -999001  1000000     - /usr/lib/wsl/drivers
/dev/sdd       67108864   41468 67067396    1% /
none            2027470      23  2027447    1% /mnt/wslg
none            2027470       6  2027464    1% /usr/lib/wsl/lib
rootfs          2026031      12  2026019    1% /init
none            2027470     566  2026904    1% /run
none            2027470       2  2027468    1% /run/lock
none            2027470       1  2027469    1% /run/shm
none            2027470      52  2027418    1% /mnt/wslg/versions.txt
none            2027470      52  2027418    1% /mnt/wslg/doc
C:\                 999 -999001  1000000     - /mnt/c
snapfuse             29      29        0  100% /snap/bare/5
snapfuse          14270   14270        0  100% /snap/core22/2292
snapfuse          1427

wsl: Failed to translate 'Z:\home\usama\linux_ai_systems_lab'
